# Supply Chain Monitoring


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, current_date, when

spark = SparkSession.builder.appName("SupplyChainMonitoringWeek3").getOrCreate()

In [2]:
from google.colab import files
uploaded = files.upload()
data_dir = "/content"


Saving orders.csv to orders.csv
Saving suppliers.csv to suppliers.csv


In [3]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/content/orders.csv")
suppliers = spark.read.option("header", True).option("inferSchema", True).csv("/content/suppliers.csv")

orders.printSchema()
suppliers.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- expected_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- supplier_id: integer (nullable = true)
 |-- supplier_name: string (nullable = true)



In [4]:
orders = orders.withColumn("order_date", col("order_date").cast("date")) \
               .withColumn("expected_delivery_date", col("expected_delivery_date").cast("date")) \
               .withColumn("actual_delivery_date", col("actual_delivery_date").cast("date"))

shipment_details = orders.join(suppliers, on="supplier_id", how="left")

shipment_details = shipment_details.withColumn(
    "delay_days",
    when(
        col("actual_delivery_date").isNotNull(),
        datediff(col("actual_delivery_date"), col("expected_delivery_date"))
    ).otherwise(
        datediff(current_date(), col("expected_delivery_date"))
    )
).withColumn(
    "is_delayed",
    when(col("delay_days") > 0, True).otherwise(False)
)

shipment_details.select(
    "order_id", "supplier_name", "order_date",
    "expected_delivery_date", "actual_delivery_date", "delay_days", "is_delayed"
).show(10, truncate=False)

+--------+-----------------------+----------+----------------------+--------------------+----------+----------+
|order_id|supplier_name          |order_date|expected_delivery_date|actual_delivery_date|delay_days|is_delayed|
+--------+-----------------------+----------+----------------------+--------------------+----------+----------+
|2001    |Southern Components Ltd|2026-09-03|2026-09-09            |2026-09-08          |-1        |false     |
|2002    |Metro Packaging Co     |2026-09-05|2026-09-12            |NULL                |51        |true      |
|2003    |Greenline Electronics  |2026-09-07|2026-09-12            |NULL                |51        |true      |
|2004    |Reliable Plastics      |2026-09-09|2026-09-15            |2026-09-14          |-1        |false     |
|2005    |Prime Industrial Supply|2026-09-11|2026-09-18            |NULL                |45        |true      |
|2006    |East Coast Logistics   |2026-09-13|2026-09-18            |NULL                |45        |true

In [7]:
from pyspark.sql.functions import col, sum as _sum, avg, count

delayed_shipments = shipment_details.filter(col("is_delayed") == True)
delayed_by_supplier = shipment_details.groupBy("supplier_id", "supplier_name").agg(
    count("order_id").alias("total_orders"),
    _sum(col("is_delayed").cast("integer")).alias("delayed_orders"),
    avg("delay_days").alias("average_delay_days")
)

delayed_shipments.show(truncate=False)
delayed_by_supplier.show(truncate=False)

+-----------+--------+---------------------+----------+----------+----------------------+--------------------+--------+------------+-----------------------+----------+----------+
|supplier_id|order_id|product_name         |sku       |order_date|expected_delivery_date|actual_delivery_date|quantity|order_status|supplier_name          |delay_days|is_delayed|
+-----------+--------+---------------------+----------+----------+----------------------+--------------------+--------+------------+-----------------------+----------+----------+
|202        |2002    |Shipping Box - Medium|SKU-BOX-02|2026-09-05|2026-09-12            |NULL                |124     |Delayed     |Metro Packaging Co     |51        |true      |
|203        |2003    |Wireless Mouse       |SKU-MSE-03|2026-09-07|2026-09-12            |NULL                |161     |In Transit  |Greenline Electronics  |51        |true      |
|205        |2005    |Keyboard             |SKU-KBD-05|2026-09-11|2026-09-18            |NULL            

In [8]:
delayed_shipments.write.mode("overwrite").option("header", True).csv("../outputs/delayed_shipments")
delayed_by_supplier.write.mode("overwrite").option("header", True).csv("../outputs/delayed_orders_by_supplier")

In [9]:
!zip -r my_output.zip /outputs

  adding: outputs/ (stored 0%)
  adding: outputs/delayed_shipments/ (stored 0%)
  adding: outputs/delayed_shipments/part-00000-3af17d19-0297-4a54-99e3-e77ceedd6657-c000.csv (deflated 69%)
  adding: outputs/delayed_shipments/._SUCCESS.crc (stored 0%)
  adding: outputs/delayed_shipments/_SUCCESS (stored 0%)
  adding: outputs/delayed_shipments/.part-00000-3af17d19-0297-4a54-99e3-e77ceedd6657-c000.csv.crc (stored 0%)
  adding: outputs/delayed_orders_by_supplier/ (stored 0%)
  adding: outputs/delayed_orders_by_supplier/part-00000-4a68a5b5-e998-4e21-9407-87489e3e4cd7-c000.csv (deflated 33%)
  adding: outputs/delayed_orders_by_supplier/._SUCCESS.crc (stored 0%)
  adding: outputs/delayed_orders_by_supplier/_SUCCESS (stored 0%)
  adding: outputs/delayed_orders_by_supplier/.part-00000-4a68a5b5-e998-4e21-9407-87489e3e4cd7-c000.csv.crc (stored 0%)


In [10]:
from google.colab import files
files.download('my_output.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
spark.stop()